# YT Agent — Colab GPU backend

This notebook spins up the FastAPI backend on a free Colab GPU runtime, exposes it via a Cloudflare quick tunnel, and registers itself in the R2 registry so your Vercel frontend can find it.

## Before you start
1. **Runtime → Change runtime type → T4 GPU** (free).
2. Open **🔑 Secrets** (left sidebar key icon) and add these **5 required** Cloudflare R2 secrets:
   - `R2_ACCOUNT_ID`
   - `R2_ACCESS_KEY_ID`
   - `R2_SECRET_ACCESS_KEY`
   - `R2_BUCKET` (e.g. `yt-agent`)
   - `R2_PUBLIC_URL` (your `pub-xxx.r2.dev` URL — no trailing slash)

   **Optional** (enables Hostinger archive overflow when R2 hits 7 GB):
   - `SFTP_HOST`, `SFTP_PORT`, `SFTP_USER`, `SFTP_PASS`, `SFTP_BASE_DIR`, `PUBLIC_BASE_URL`

   Everything else (NIM, Shutterstock, Pexels, etc.) is pulled from the central `keys.json` on R2 that the dashboard manages — you don't paste them here.
3. **Toggle notebook access ON** for each secret (the X icons → green dots).
4. Run cells top to bottom. The last cell is a blocking server loop — leave it running.

When the tunnel URL appears, the registry update is automatic. Your Vercel dashboard will pick this backend up within ~30s.

In [ ]:
# 1) System deps
import subprocess, sys, os
subprocess.check_call(["apt-get", "-qq", "update"])
subprocess.check_call(["apt-get", "-qq", "install", "-y", "ffmpeg"])
print("ffmpeg:", subprocess.check_output(["ffmpeg", "-version"]).decode().splitlines()[0])

In [ ]:
# 2) Clone the repo (or pull latest if already present)
REPO_URL = "https://github.com/Ahsan3301/yt_agent.git"
BRANCH   = "main"
import os, subprocess
if not os.path.exists('/content/yt_agent'):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, '/content/yt_agent'])
else:
    subprocess.check_call(['git', '-C', '/content/yt_agent', 'pull', '--ff-only'])
os.chdir('/content/yt_agent')
print('repo at:', os.getcwd())

In [ ]:
# 3) Python deps (includes boto3 for R2 + paramiko for SFTP archive)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

In [ ]:
# 4) BOOTSTRAP secrets only — the rest are pulled from keys.json on R2
#    when the backend starts.
#
#    REQUIRED in the 🔑 Secrets panel (left rail key icon):
#      R2_ACCOUNT_ID · R2_ACCESS_KEY_ID · R2_SECRET_ACCESS_KEY
#      R2_BUCKET · R2_PUBLIC_URL
#
#    OPTIONAL (enables overflow archive when R2 fills up):
#      SFTP_HOST · SFTP_PORT · SFTP_USER · SFTP_PASS · SFTP_BASE_DIR
#      PUBLIC_BASE_URL
#
#    Set NIM / Shutterstock / Pexels / Pixabay / Coverr / Groq via the
#    DASHBOARD's "API Keys" page — Vercel writes them to R2 and this
#    Colab pulls them on startup automatically.
from google.colab import userdata
import os

REQUIRED = ['R2_ACCOUNT_ID', 'R2_ACCESS_KEY_ID', 'R2_SECRET_ACCESS_KEY',
            'R2_BUCKET', 'R2_PUBLIC_URL']
OPTIONAL = ['R2_MAX_GB',
            'SFTP_HOST', 'SFTP_PORT', 'SFTP_USER', 'SFTP_PASS', 'SFTP_BASE_DIR',
            'PUBLIC_BASE_URL',
            'INSTANCE_LABEL', 'INSTANCE_TIER']

missing = []
for k in REQUIRED + OPTIONAL:
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = str(v)
    except Exception:
        if k in REQUIRED:
            missing.append(k)
        continue
    if k in REQUIRED and not os.environ.get(k):
        missing.append(k)

# Sensible default: Colab = GPU tier (preferred over CPU fallback).
os.environ.setdefault('INSTANCE_TIER', 'gpu')
os.environ.setdefault('INSTANCE_LABEL', 'colab-gpu')

if missing:
    print('MISSING REQUIRED BOOTSTRAP SECRETS:', ', '.join(missing))
    print('Add them in the 🔑 Secrets panel and toggle notebook access ON, then re-run this cell.')
else:
    print('Bootstrap secrets loaded.')
    print('  R2 primary  :', os.environ.get('R2_BUCKET'), 'at', os.environ.get('R2_PUBLIC_URL'))
    sec_ok = bool(os.environ.get('SFTP_HOST') and os.environ.get('PUBLIC_BASE_URL'))
    print('  SFTP archive:', 'configured' if sec_ok else 'not set (no overflow migration)')
    print('  API keys will be pulled from keys.json on R2.')

In [ ]:
# 5) Cloudflare quick tunnel — free, no auth. Returns a stable URL
#    until the Colab session ends.
import os, subprocess, time, re, threading

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.check_call([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared',
    ])
    subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])

# Start the tunnel to localhost:8000 (where uvicorn will run in the next cell)
tunnel_log = '/tmp/cloudflared.log'
subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
     '--logfile', tunnel_log, '--loglevel', 'info'],
    stdout=open(tunnel_log + '.stdout', 'w'),
    stderr=subprocess.STDOUT,
)

# Wait for the URL to show up in the log (~5-10s)
url = None
for _ in range(40):
    time.sleep(0.5)
    if not os.path.exists(tunnel_log):
        continue
    with open(tunnel_log) as f:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
        if m:
            url = m.group(0)
            break
if not url:
    raise RuntimeError('cloudflared did not produce a URL — check /tmp/cloudflared.log')

os.environ['PUBLIC_BACKEND_URL'] = url
print('Public backend URL:', url)

In [ ]:
# 6) Boot the backend. This cell BLOCKS — keep it running for the session.
#    The heartbeat (registry.start) auto-registers PUBLIC_BACKEND_URL into
#    your R2 registry.json every 30s.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])